In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc 
import anndata as ad
import h5py 
import glob
import matplotlib.pyplot as plt
import glob
import os
import seaborn as sns

In [ ]:
# set base path 
base_path = '/home/EOCRC_atlas/'

In [ ]:
# load data 
adata = sc.read_h5ad(os.path.join(base_path, 'data/all_samples_processed_withTier2annotation.h5ad'))

In [ ]:
# QC table per patient 
summary_per_sample = adata.obs.groupby('FRID').agg(
    total_cells=('n_genes_by_counts', 'count'),
    median_genes_per_cell=('n_genes_by_counts', 'median'),
    median_umis_per_cell=('total_counts', 'median'),
    median_mito_percent=('pct_counts_mt', 'median')
).reset_index()

print(summary_per_sample['total_cells'].median())
summary_per_sample.to_csv(os.path.join(base_path, "results/2025-09-26_YOCRC_annotations/QCs_by_sample.csv"))
summary_per_sample

In [ ]:
# make table of celltype counts per sample 
celltype_counts = (adata.obs.groupby(['FRID', 'Annotation_Tier2']).size().reset_index(name='n_cells'))
celltype_counts

In [ ]:
# convert to wide format 
celltype_wide = (
    celltype_counts
    .pivot(index='FRID', columns='Annotation_Tier2', values='n_cells')
    .fillna(0)  
    .reset_index()
)
celltype_wide

In [ ]:
# merge with cell type QCs 
final_summary = summary_per_sample.merge(celltype_wide,on='FRID',how='left')
final_summary.to_csv(os.path.join(base_path, f'results/sample_level_QC_metrics_2-24-26.csv'))

In [ ]:
# create bar plot showing cell counts by Tier1 cell type 
cell_counts = adata.obs.groupby(['FRID', 'Annotation_Tier1']).size().unstack(fill_value=0)

# Get unique cell types
cell_types = adata.obs['Annotation_Tier1'].unique()

# Create a subplot for each cell type
n_cells = len(cell_types)
fig, axes = plt.subplots(n_cells, 1, figsize=(45, 20), sharex=True)

if n_cells == 1:
    axes = [axes]

# Loop through each cell type and plot
for i, cell_type in enumerate(cell_types):
    # Get the counts for the current cell type
    counts = cell_counts[cell_type]

    # Plot the bar plot
    axes[i].bar(counts.index, counts.values, color='blue')

    # Customize the individual subplot
    axes[i].set_ylabel(f'{cell_type} cells')
    axes[i].tick_params(axis='x', rotation=90)
    
    # Set x-axis limits to match the patient indices and remove extra white space
    axes[i].set_xlim(-0.5, len(counts) - 0.5)

# Set common labels
axes[-1].set_xlabel('Sample')

# Adjust layout to remove margins
plt.subplots_adjust(left=0.1, right=0.9, top=0.95, bottom=0.05)

plt.savefig(os.path.join(base_path, "results/EOCRC_annotations/celltype_counts_by_sample.pdf"), dpi=600, bbox_inches='tight')

# Adjust layout for clarity
plt.tight_layout()
plt.show()

In [ ]:
# make UMAP plots for epithelial cells 
# load data 
adata = sc.read_h5ad(os.path.join(base_path, 'data/yocrc_Epithelial_annotation_noHarmony.h5ad'))

# setup 
colors = ['#000000', '#E69F00', '#56B4E9', '#009E73', '#F0E442', '#0072B2', '#D55E00', '#CC79A7']
sc.set_figure_params(figsize=(4, 4))

# Plot DECADE
fig, ax = plt.subplots()
sc.pl.umap(adata, color='Decade', size=1, ax=ax, show=True, palette=colors)
fig.savefig(os.path.join(base_path, 'results/EOCRC_annotations/Decade_UMAP_epiOonly.pdf'), dpi=600, bbox_inches='tight')
plt.close(fig)

# Plot AGE COHORT
fig, ax = plt.subplots()
sc.pl.umap(adata, color='Cohort', size=1, ax=ax, show=True, palette=colors[5:7])
fig.savefig(os.path.join(base_path, 'results/EOCRC_annotations/Cohort_UMAP_epiOonly.pdf'), dpi=600, bbox_inches='tight')
plt.close(fig)

# Plot MSI COHORT
fig, ax = plt.subplots()
sc.pl.umap(adata, color='MSI_v2', size=1, ax=ax, show=True, palette=colors[0:3])
fig.savefig(os.path.join(base_path, 'results/EOCRC_annotations/MSI_UMAP_epiOonly.pdf'), dpi=600, bbox_inches='tight')
plt.close(fig)

# Plot SIDE COHORT
fig, ax = plt.subplots()
sc.pl.umap(adata, color='Sidedness', size=1, ax=ax, show=True, palette=colors[3:6])
fig.savefig(os.path.join(base_path, 'results/EOCRC_annotations/Sidedness_UMAP_epiOonly.pdf'), dpi=600, bbox_inches='tight')
plt.close(fig)

In [ ]:
# Make a violin plot of nGene, nUMI, % mito by decade

# load data 
adata = sc.read_h5ad(os.path.join(base_path, 'data/all_samples_processed_withTier2annotation.h5ad'))

# set colors 
colors = ['#E69F00', '#56B4E9', '#009E73', '#F0E442', '#0072B2', '#D55E00', '#CC79A7']

fig, ax = plt.subplots()
sc.pl.violin(adata, ['n_genes_by_counts'], groupby='Decade', ax=ax,
             use_raw=True, rotation=45, stripplot=False, inner="quartile", palette = colors)
fig.savefig(os.path.join(base_path, 'results/EOCRC_annotations/Decade_nGene.pdf'), dpi=600, bbox_inches='tight')
plt.close(fig)

fig, ax = plt.subplots()
sc.pl.violin(adata, ['total_counts'], groupby='Decade', ax=ax,
             use_raw=True, rotation=45, stripplot=False, inner="quartile", palette = colors)
fig.savefig(os.path.join(base_path, 'results/EOCRC_annotations/Decade_nUMI.pdf'), dpi=600, bbox_inches='tight')
plt.close(fig)

fig, ax = plt.subplots()
sc.pl.violin(adata, ['pct_counts_mt'], groupby='Decade', ax=ax,
             use_raw=True, rotation=45, stripplot=False, inner="quartile", palette = colors)
fig.savefig(os.path.join(base_path, 'results/EOCRC_annotations/Decade_pctMito.pdf'), dpi=600, bbox_inches='tight')
plt.close(fig)

In [ ]:
# make violin plot showing nuclei per sample grouped by decade 
sample_df = (adata.obs.groupby(['FRID', 'Decade']).size().reset_index(name='nuclei_count'))
colors = ['#E69F00', '#56B4E9', '#009E73', '#F0E442', '#0072B2', '#D55E00', '#CC79A7']

fig, ax = plt.subplots(figsize=(4, 4))
sns.violinplot(data=sample_df,x='Decade',y='nuclei_count',ax=ax,palette=colors,inner="quartile",cut=0)

plt.xticks(rotation=45)
ax.set_ylabel('# nuclei/sample')
ax.set_xlabel('Decade')
sns.despine()  # Removes the top and right border lines for a clean look

fig.savefig(os.path.join(base_path, 'results/EOCRC_annotations/Decade_nucleiCount_QC.pdf'), dpi=600, bbox_inches='tight')
plt.show(fig)
plt.close(fig)

In [ ]:
# get cell type counts and percentages for manuscript 
cell_counts = adata.obs['Annotation_Tier1'].value_counts()
cell_percentages = (cell_counts / cell_counts.sum()) * 100
cell_summary = pd.DataFrame({
    'Count': cell_counts,
    'Percent': cell_percentages.round(2)
})
print(cell_summary)

In [ ]:
# check mito percentage per annotation 
median_mito = adata.obs.groupby('Annotation_Tier2')['pct_counts_mt'].median()
print(median_mito)

In [ ]:
# Make  dot plot based on final annotation for ED figs
# Calculate clsuter marker genes 
sc.tl.rank_genes_groups(adata, "Annotation_Tier2", method="wilcoxon", use_raw = True)

# save marker genes 
df_all = pd.DataFrame()
clusters = np.unique(adata.obs['Annotation_Tier2'])
for i in clusters:
    
    df = sc.get.rank_genes_groups_df(adata, group = i)

    #order by zscore
    df['abs_logFC'] = np.absolute(df['logfoldchanges'])
    df = df[['names', 'scores', 'logfoldchanges', 'abs_logFC', 'pvals', 'pvals_adj']]
    print(i)
    print(df)
    
    i = i.replace(" / ", "_")
    i = i.replace(" ", "_")
        
    tmp=df['names'][0:100]
    df_all = pd.concat([df_all, tmp], axis=1)
df_all.columns=clusters
df_all.to_csv(os.path.join(base_path, f'results/annotConfirm/Epi_Annotation_Tier2_markerGenes_top100.csv'))

In [ ]:
df_all = pd.read_csv(os.path.join(base_path, f'results/annotConfirm/Epi_Annotation_Tier2_markerGenes_top100.csv'))
for i in df_all.columns:
    print(i)
    sc.pl.umap(adata, color=df_all[str(i)][0:30], ncols=10)

In [ ]:
df_all = df_all[['CEACAM1 colonocyte-like',
    'Enteroendocrine-like', 
    'LGR5 stem cell-like',
    'MT-Ribo-hi epithelial',
    'MUC2 goblet-like']]
df_all

In [ ]:
genes = []
for i in range(5):
    genes.extend(df_all[df_all.columns[i]][0:30].values)

In [ ]:
# dot plot of epi markers 
row_order = [
    'CEACAM1 colonocyte-like',
    'Enteroendocrine-like', 
    'LGR5 stem cell-like',
    'MT-Ribo-hi epithelial',
    'MUC2 goblet-like',    
]
adata_sub = adata[~adata.obs['Annotation_Tier2'].isin(['Patient-specific', 'Mixed - epithelial'])].copy()
adata_sub.obs['Annotation_Tier2'] = adata_sub.obs['Annotation_Tier2'].astype('category')
adata_sub.obs['Annotation_Tier2'] = adata_sub.obs['Annotation_Tier2'].cat.reorder_categories(row_order, ordered=True)

sc.set_figure_params(figsize=(34, 3))
fig, ax = plt.subplots()
sc.pl.dotplot(adata_sub, var_names=genes, groupby='Annotation_Tier2', use_raw=True, standard_scale='var',
              dot_max=1, dot_min=0, ax=ax)
fig.savefig(os.path.join(base_path, f'results/annotConfirm/Epithelial_Annotation_Tier2_markers_dotplot.pdf'), dpi=600, bbox_inches='tight')
plt.close(fig)

In [ ]:
# DEG dot plots 
adata = sc.read_h5ad(os.path.join(base_path, 'data/all_samples_processed_withTier2annotation.h5ad'))
adata = adata[adata.obs['MSI_v2']=='MSS: STABLE']

adata.obs['Decade_collapsed'] = adata.obs['Decade'].astype(str)
adata.obs.loc[adata.obs['Decade_collapsed'] == "20-29", 'Decade_collapsed'] = "<40"
adata.obs.loc[adata.obs['Decade_collapsed'] == "30-39", 'Decade_collapsed'] = "<40"
adata.obs['Decade_collapsed'] = adata.obs['Decade_collapsed'].astype('category')
adata.obs['Decade_collapsed'] = (adata.obs['Decade_collapsed'].cat.set_categories(['<40', '40-49', '50-59', '60-69', '70-79', '80-89', '90-99'], ordered=True))

clusters = ['LGR5_stem_cell-like', 'CEACAM1_colonocyte-like', 'MT-Ribo-hi_epithelial', 'Fibroblast']

add_young = ['AC007255.1', 'AC022101.1', 'AC119150.1', 'LINC01344', 'PTCHD1-AS', 'LINC01524', 'CCDC144NL-AS1', 
              'FN1', 'LINC00607', 'RPL39L', 'FSIP1', 'SLC26A2','CHODL']
add_old = ['COL16A1', 'NPAS3', 'NDRG1', 'SLC2A3', 'TAOK3']

for i in clusters: 
    print(i)
    tmp = adata[adata.obs['Annotation_Tier2']==i.replace("_", " ")]
    print(f"Number of cells: {tmp.shape[0]}")

    genes = pd.read_csv(
        os.path.join(base_path, f'results/YOCRC_DEGs/DESeq2_cluster_{i}_AgeScaled__MSS_sideTherapyCov_09-19-25_1pctExpressed_contScaleAge_YOCRC_filtered.csv'))
    genes_sorted = genes.sort_values(by="log2FoldChange", ascending=True)
    neg_genes = genes_sorted[genes_sorted["log2FoldChange"] < 0].head(15)["Gene"].tolist()
    pos_genes = genes_sorted[genes_sorted["log2FoldChange"] > 0].tail(15)["Gene"].tolist()

    repeat_neg = list(set(genes_sorted.loc[genes_sorted["log2FoldChange"] < 0, "Gene"]) & set(add_young) - set(neg_genes))
    repeat_pos = list(set(genes_sorted.loc[genes_sorted["log2FoldChange"] > 0, "Gene"]) & set(add_old) - set(pos_genes))

    genes_plot = neg_genes + repeat_neg + pos_genes + repeat_pos 

    sc.set_figure_params(figsize=(max(2.5, len(genes_plot)*.3) , 3))
    fig, ax = plt.subplots()
    sc.pl.dotplot(tmp, genes_plot, use_raw=True, groupby="Decade_collapsed", dot_max=1, dot_min=0, 
                 standard_scale='var', mean_only_expressed=False, smallest_dot=10, ax=ax, show=False)
    plt.show()
    fig.savefig(os.path.join(base_path, f'results/YOCRC_DEGs/DEG_dotplot_{i}_ageCont_MSS_topGenes_textGenes.pdf'), 
                dpi=600, bbox_inches='tight')
    plt.close(fig)

In [ ]:
# print atlas statistics 
adata = sc.read_h5ad(os.path.join(base_path, 'data/all_samples_processed_withTier2annotation.h5ad'))
print(f"median nGene : {adata.obs.n_genes_by_counts.median()}\nmedian nUMI : {adata.obs.total_counts.median()}\n" \
      f"median pMT {adata.obs.pct_counts_mt.median()}")

unique_frids_per_cohort = (
    adata.obs.groupby('Cohort')['FRID']
    .nunique()
    .reset_index()
    .rename(columns={'FRID': 'n_unique_FRIDs'})
)
print(unique_frids_per_cohort)

unique_frids_per_decade = (
    adata.obs.groupby('Decade')['FRID']
    .nunique()
    .reset_index()
    .rename(columns={'FRID': 'n_unique_FRIDs'})
)
print(unique_frids_per_decade)